# 01 - Kham Pha Du Lieu (EDA)

**Do An: Su dung Color Correlogram cho nhan dang anh bang hoc may**

Notebook nay kham pha dataset Corel-1K:
- Hien thi anh mau tu moi lop
- Thong ke so luong anh
- Phan tich phan bo mau sac

In [ ]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Them duong dan src
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from preprocessing import load_dataset, convert_to_hsv, quantize_colors_hsv

# Duong dan dataset
DATA_DIR = os.path.join('..', 'data', 'corel-1k')
print(f'Dataset directory: {os.path.abspath(DATA_DIR)}')
print(f'Exists: {os.path.exists(DATA_DIR)}')

## 1. Tai va kiem tra dataset

In [ ]:
# Tai dataset
images, labels, paths = load_dataset(DATA_DIR)

# Thong ke
unique_labels = sorted(set(labels))
print(f'\nTong so anh: {len(images)}')
print(f'So lop: {len(unique_labels)}')
print(f'Cac lop: {unique_labels}')

In [ ]:
# Dem so anh moi lop
from collections import Counter

label_counts = Counter(labels)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(label_counts.keys(), label_counts.values(), color='steelblue', edgecolor='black')
ax.set_xlabel('Lop (Class)', fontsize=12)
ax.set_ylabel('So luong anh', fontsize=12)
ax.set_title('Phan bo so luong anh theo lop', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')

for bar, count in zip(bars, label_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Hien thi anh mau tu moi lop

In [ ]:
# Hien thi 3 anh mau tu moi lop
n_samples = 3
n_classes = len(unique_labels)

fig, axes = plt.subplots(n_classes, n_samples, figsize=(4*n_samples, 3*n_classes))

for i, class_name in enumerate(unique_labels):
    # Lay cac anh thuoc lop nay
    class_indices = [j for j, l in enumerate(labels) if l == class_name]
    sample_indices = class_indices[:n_samples]
    
    for j, idx in enumerate(sample_indices):
        img_rgb = cv2.cvtColor(images[idx], cv2.COLOR_BGR2RGB)
        axes[i, j].imshow(img_rgb)
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(f'{class_name}', fontsize=12, fontweight='bold')

plt.suptitle('Anh mau tu moi lop trong dataset', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 3. Phan tich phan bo mau sac

In [ ]:
# Phan tich phan bo mau HSV trung binh cho moi lop
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, class_name in enumerate(unique_labels):
    class_indices = [j for j, l in enumerate(labels) if l == class_name]
    
    # Lay tat ca anh cua lop nay
    h_values = []
    for idx in class_indices[:20]:  # Lay 20 anh de tinh toan nhanh
        img_hsv = cv2.cvtColor(images[idx], cv2.COLOR_BGR2HSV)
        h_values.extend(img_hsv[:, :, 0].ravel().tolist())
    
    axes[i].hist(h_values, bins=36, range=(0, 180), color='coral', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{class_name}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Hue')
    axes[i].set_ylabel('Count')

plt.suptitle('Phan bo kenh Hue (H) theo tung lop', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Phan tich mau trung binh cua moi lop
avg_colors = []
for class_name in unique_labels:
    class_indices = [j for j, l in enumerate(labels) if l == class_name]
    mean_bgr = np.mean([images[idx].mean(axis=(0,1)) for idx in class_indices[:20]], axis=0)
    avg_colors.append(mean_bgr)

# Hien thi mau trung binh
fig, ax = plt.subplots(figsize=(12, 3))
for i, (name, bgr) in enumerate(zip(unique_labels, avg_colors)):
    rgb = bgr[::-1] / 255.0  # BGR -> RGB, normalize
    ax.bar(i, 1, color=rgb, edgecolor='black', width=0.8)
    ax.text(i, -0.15, name, ha='center', fontsize=9, rotation=45)

ax.set_xlim(-0.5, len(unique_labels) - 0.5)
ax.set_ylim(-0.5, 1.2)
ax.set_title('Mau sac trung binh cua moi lop', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print('\nNhan xet: Cac lop co mau sac trung binh khac nhau ro rang,')
print('dieu nay cho thay Color Correlogram co the hieu qua cho bai toan nay.')